In [3]:
# In this version of OGA with random dictionaries, we use QMC to evaluate the loss function. 
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import time
import sys
import os 
from scipy.sparse import linalg
from pathlib import Path
import itertools
if torch.cuda.is_available():  
    device = "cuda" 
else:  
    device = "cpu" 
pi = torch.tensor(np.pi,dtype=torch.float64)
torch.set_default_dtype(torch.float64)

class model(nn.Module):
    """ ReLU k shallow neural network
    Parameters: 
    input size: input dimension
    hidden_size1 : number of hidden layers 
    num_classes: output classes 
    k: degree of relu functions
    """
    def __init__(self, input_size, hidden_size1, num_classes,k = 1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, num_classes,bias = False)
        self.k = k 
    def forward(self, x):
        u1 = self.fc2(F.relu(self.fc1(x))**self.k)
        return u1

def generate_relu_dict4plusD_fromSd(dim, s,N0):

    samples = torch.randn(s*N0,dim+1)  

    samples = samples/torch.norm(samples, dim = 1,keepdim=True)

    return samples.to(device)

def generate_relu_dict4plusD_QMC(dim, s,N0):

    samples = torch.rand(s*N0,dim)  

    # for i in range(s-1):
        # samples = torch.cat([samples,Sob.draw(N0).double()],0)
    # Form the transformation matrix and shift vector 
    diagonal = torch.ones(dim)*pi  
    diagonal[-1] =  2*dim**0.5
    diagonal[-2] = 2*pi 
    T = torch.diag(diagonal)

    shift = torch.zeros(dim)
    shift[-1] = -dim**0.5 
    samples = samples@T + shift 

    Wb_tensor = torch.ones(s*N0,dim+1) # each neuron parameter stored in rows  
    for i in range(dim): # 0, 1, ... dim-1 
        for j in range(i+1): # 0, 1, ... i 
            if i == 0: 
                Wb_tensor[:,i] = Wb_tensor[:,i]*torch.cos(samples[:,j])
            if i == (dim - 1):
                if j != i:
                    Wb_tensor[:,i] = Wb_tensor[:,i] * torch.sin(samples[:,j]) 
            if i != 0 and i != (dim - 1): 
                if j != i: 
                    Wb_tensor[:,i] = Wb_tensor[:,i] * torch.sin(samples[:,j]) 
                else: 
                    Wb_tensor[:,i] = Wb_tensor[:,i] * torch.cos(samples[:,j]) 
            
    Wb_tensor[:,dim] = samples[:,-1] 

    return Wb_tensor.to(device)

def initialize_model_random(my_model,dim):
    
    # (w,b) ~ U(S^d) random 
    neuron_nums = my_model.fc1.bias.size(0)
    points = torch.randn(neuron_nums,dim+1)
    points = points/torch.norm(points, dim=1, keepdim=True)
    my_model.fc1.weight.data[:,:] = points[:,0:dim]
    my_model.fc1.bias.data[:] = points[:,dim]  
    return my_model 

def remove_redundant_neuron(my_model, dims = 3, choice = 2): 
    ##  choice 1:  [0,1]^d, choice 2: [-1,1]^d
    def create_mesh_grid(dims, pts):
        mesh = torch.tensor(list(itertools.product(pts,repeat=dims)))
        vertices = mesh.reshape(len(pts) ** dims, -1) 
        return vertices
    counter = 0 
    # positions = torch.tensor([[0.,0.],[0.,1.],[1.,1.],[1.,0.]])
    # pts = torch.tensor([0.,1.]) # for domain [0,1]^d 
    if choice == 1: 
        pts = torch.tensor([0.,1.])
    elif choice == 2:
        pts = torch.tensor([-1.,1.])# for domain [-1,1]^d 
    elif choice == 3:
        pts = torch.tensor([-1./2,1./2])# for domain [-1,1]^d 
    positions = create_mesh_grid(dims,pts) 
    neuron_num = my_model.fc1.bias.size(0)
    relu_k = my_model.k 
    recorded_neurons = []
    for i in range(neuron_num): 
        w = my_model.fc1.weight.data[i:i+1,:]
        b = my_model.fc1.bias.data[i]
        values = torch.matmul(positions,w.T)
        left_end = - torch.max(values)
        right_end = - torch.min(values)
        offset = (right_end - left_end)/50
        if b > left_end + offset/2 and b < right_end - offset/2: 
            recorded_neurons.append((w, b))
        elif b >= right_end - offset/2 and counter < (dims+1):
            recorded_neurons.append((w, b))
            counter += 1

           
    new_neuron_num = len(recorded_neurons)
    
    num_to_add = neuron_num - new_neuron_num
    counter2 = 0
    while counter2 < num_to_add: 
        sample = torch.randn(1,dims+1)  
        sample = sample/torch.norm(sample, dim = 1,keepdim=True)
        w = sample[0:1,:dims]
        b = sample[0,dims]
        values = torch.matmul(positions,w.T)
        left_end = - torch.max(values)
        right_end = - torch.min(values)
        offset = (right_end - left_end)/50
        if b > left_end + offset/2 and b < right_end - offset/2: 
            recorded_neurons.append((w, b))
            counter2 += 1 

#     new_model = model(dims, new_neuron_num, 1, k=relu_k).to(device)
    new_model = model(dims, new_neuron_num + counter2, 1, k=relu_k).to(device)

    for i, (w, b) in enumerate(recorded_neurons):
        new_model.fc1.weight.data[i:i+1,:] = w
        new_model.fc1.bias.data[i] = b
    print("Number of neurons removed: ", neuron_num - new_neuron_num)
    print("Number of neurons left: ", new_neuron_num)  
    print("after replenishing neurons: ", new_neuron_num + counter2)
    return new_model

def MonteCarlo_Sobol_dDim_weights_points(M ,d = 4,bl = -1,ur = 1):
    
    length = ur - bl
    vol = length ** d 
    Sob_integral = torch.quasirandom.SobolEngine(dimension =d, scramble= False, seed=None) 
    integration_points = Sob_integral.draw(M).double() 
    integration_points = integration_points.to(device) * (2 * length) - length
    weights = torch.ones(M,1).to(device)/M * vol 
    return weights.to(device), integration_points.to(device) 


def minimize_linear_layer_explicit_assemble(model,target,weights, integration_points,solver="direct",memory=2**27):
    """
    """
    start_time = time.time() 
    w = model.fc1.weight.data 
    b = model.fc1.bias.data 
    
    # new batched operation 
    n = b.size(0)
    M = integration_points.size(0)
    
    total_size = n * M # memory, number of floating numbers 
    num_batch = total_size//memory + 1 # divide according to memory
    batch_size = M//num_batch
    print("num batch: ", num_batch )
    start_ind = 0
    end_ind = 0 
    jac = torch.zeros(b.size(0),b.size(0)).to(device)
    rhs = torch.zeros(b.size(0),1).to(device)
#     print("mat assemble, number batches: ",num_batch)
    for j in range(0,M,batch_size): 
        end_ind = j + batch_size
        basis_value_col = F.relu(integration_points[j:end_ind] @ w.t()+ b)**(model.k) 
        weighted_basis_value_col = basis_value_col * weights[j:end_ind] 
        jac += weighted_basis_value_col.t() @ basis_value_col 
        rhs += weighted_basis_value_col.t() @ (target(integration_points[j:end_ind,:])) 
        
    print("jac: ", jac.device)
    print("assembling the matrix time taken: ", time.time()-start_time) 
    start_time = time.time()    
    if solver == "cg": 
        sol, exit_code = linalg.cg(np.array(jac.detach().cpu()),np.array(rhs.detach().cpu()),tol=1e-12)
        sol = torch.tensor(sol).view(1,-1)
    elif solver == "direct": 
#         sol = np.linalg.inv( np.array(jac.detach().cpu()) )@np.array(rhs.detach().cpu())
        sol = (torch.linalg.solve( jac.detach(), rhs.detach())).view(1,-1)
    elif solver == "ls":
        sol = (torch.linalg.lstsq(jac.detach().cpu(),rhs.detach().cpu(),driver='gelsd').solution).view(1,-1)
        # sol = (torch.linalg.lstsq(jac.detach(),rhs.detach()).solution).view(1,-1) # gpu/cpu, driver = 'gels', cannot solve singular
    print("solving Ax = b time taken: ", time.time()-start_time)
    return sol 


def OGAL2FittingReLU4Dplus_QMC(my_model,target,s,N0,num_epochs, M, k =1, linear_solver = "direct", memory = 2**28): 
    
    """ Orthogonal greedy algorithm using 1D ReLU dictionary over [-pi,pi]
    Parameters
    ----------
    my_model: 
        nn model
    target: 
        target function
    num_epochs: int 
        number of training epochs 
    integration_intervals: int 
        number of subintervals for piecewise numerical quadrature 

    Returns
    -------
    err: tensor 
        rank 1 torch tensor to record the L2 error history  
    model: 
        trained nn model 
    """
    #Todo Done
    # samples for QMC integral
    dim = 5 
    start_time = time.time()
    # Sob_integral = torch.quasirandom.SobolEngine(dimension =4, scramble= False, seed=None) 
    # integration_points = Sob_integral.draw(M).double() 
    # integration_points = integration_points.to(device)
    integration_weights, integration_points = MonteCarlo_Sobol_dDim_weights_points(M ,d = dim) 
    integration_weights_test, integration_points_test = MonteCarlo_Sobol_dDim_weights_points(M*2 ,d = dim) 
    print("generate sob sequence:", time.time() - start_time) 

    err = torch.zeros(num_epochs+1)
    if my_model == None: 
        func_values = target(integration_points)
        num_neuron = 0

        list_b = []
        list_w = []

    else: 
        func_values = target(integration_points) - my_model(integration_points).detach()
        bias = my_model.fc1.bias.detach().data
        weights = my_model.fc1.weight.detach().data
        num_neuron = int(bias.size(0))

        list_b = list(bias)
        list_w = list(weights)
    
    # initial error Todo Done

    func_values_sqrd = func_values*func_values
    # print(func_values_sqrd.size())
    # print(gw_expand.size() ) 

    err[0]= (integration_weights.t() @ func_values_sqrd)**0.5
    all_start_time = time.time()
    
    solver = linear_solver
    print("using linear solver: ",solver)
    for i in range(num_epochs): 
#         relu_dict_parameters = generate_relu_dict4plusD_QMC(dim, s,N0).t() 
        relu_dict_parameters = generate_relu_dict4plusD_fromSd(dim, s,N0).t()
        print("epoch: ",i+1, end = '\t')
        if num_neuron == 0: 
            func_values = target(integration_points)
        else: 
            

            total_size = num_neuron * M 
            num_batch = total_size//(memory) + 1 # divide according to memory
            batch_size = M//num_batch
            end_ind = 0 

            func_values = torch.zeros(M,1).to(device)

            for j in range(0,M,batch_size): 
                end_ind = j + batch_size
                func_values[j:end_ind,:] = target(integration_points[j:end_ind,:]) - my_model(integration_points[j:end_ind,:]).detach()

        start_time = time.time() 
        
        M = integration_points.size(0)
        N = s*N0 
        output = torch.zeros(N,1)
        num_batches = (N*M)//memory + 1 # decide num_batches according to memory 
        batch_size = N//num_batches 
        print("argmax batch num, ", num_batches)
        for j in range(0,N,batch_size): 

            end_index = j + batch_size  
            basis_values_batch = (F.relu( torch.matmul(integration_points,relu_dict_parameters[0:dim, j:end_index] ) - relu_dict_parameters[dim, j:end_index])**k).T # uses broadcasting    
            output[j:end_index,0]  = (torch.abs(torch.matmul(basis_values_batch,func_values))/M)[:,0]
            
        neuron_index = torch.argmax(output.flatten())
        
#         basis_values = (F.relu( torch.matmul(integration_points,relu_dict_parameters[:,0:4].T ) - relu_dict_parameters[:,4])**k).T # uses broadcasting
#         output = torch.abs(torch.matmul(basis_values,func_values))/M # 
#         neuron_index = torch.argmax(output.flatten())
        print("argmax time taken, ", time.time() - start_time)
        
        list_w.append(relu_dict_parameters[0:dim, neuron_index]) # 
        list_b.append(-relu_dict_parameters[dim,neuron_index])
        num_neuron += 1
        my_model = model(dim,num_neuron,1,k).to(device)
        w_tensor = torch.stack(list_w, 0 ) 
        b_tensor = torch.tensor(list_b)
        my_model.fc1.weight.data[:,:] = w_tensor[:,:]
        my_model.fc1.bias.data[:] = b_tensor[:]

        start_time = time.time() 
        sol = minimize_linear_layer_explicit_assemble(my_model,target,integration_weights,integration_points, solver,memory)
        print("\t\t time taken minimize linear layer: ",time.time() - start_time) 
        my_model.fc2.weight.data[0,:] = sol[:]

        # calculate the test error
#         func_values = target(integration_points_test) - my_model(integration_points_test).detach()
        
        M2 = integration_points_test.size(0)
        total_size = num_neuron * M2 
        num_batch = total_size//memory + 1 # divide according to memory
        batch_size = M//num_batch
        end_ind = 0 

        func_values_sqrd = torch.zeros(M2,1).to(device)  
        
        for j in range(0,M2,batch_size): 
            end_ind = j + batch_size
            func_values_sqrd[j:end_ind,:] = target(integration_points_test[j:end_ind,:]) - my_model(integration_points_test[j:end_ind,:]).detach()
        
        func_values_sqrd = func_values_sqrd**2 

        #Todo Done 
        print(integration_points_test.device, func_values_sqrd.device)
        err[i+1]= ((integration_weights_test.t() @ func_values_sqrd)**0.5).data 
        print("current error: ",err[i+1]) 
    print("total duration: ",time.time() - all_start_time)
    return err, my_model


In [4]:

def show_convergence_order(err_l2,exponent,dict_size, filename,write2file = False):
    
    if write2file:
        file_mode = "a" if os.path.exists(filename) else "w"
        f_write = open(filename, file_mode)
    
    neuron_nums = [2**j for j in range(2,exponent+1)]
    err_list = [err_l2[i] for i in neuron_nums ]
    if write2file:
        f_write.write('dictionary size: {}\n'.format(dict_size))
        f_write.write("neuron num \t\t error \t\t order \t\t h10 error \\ order \n")
    print("neuron num \t\t error \t\t order")
    for i, item in enumerate(err_list):
        if i == 0: 
            print("{} \t\t {:.6f} \t\t *  \n".format(neuron_nums[i],item ) )
            if write2file: 
                f_write.write("{} \t\t {} \t\t * \t\t \n".format(neuron_nums[i],item ))
        else: 
            print("{} \t\t {:.6f} \t\t {:.6f} \n".format(neuron_nums[i],item,np.log(err_list[i-1]/err_list[i])/np.log(2) ) )
            if write2file: 
                f_write.write("{} \t\t {} \t\t {} \n".format(neuron_nums[i],item,np.log(err_list[i-1]/err_list[i])/np.log(2) ))
    if write2file:     
        f_write.write("\n")
        f_write.close()

def show_convergence_order_latex(err_l2,exponent,k=1,d=1): 
    neuron_nums = [2**j for j in range(2,exponent+1)]
    err_list = [err_l2[i] for i in neuron_nums ]
    l2_order = -1/2-(2*k + 1)/(2*d)
    print("neuron num  & \t $\\|u-u_n \\|_{{L^2}}$ & \t order $O(n^{{{:.2f}}})$  \\\\ \\hline \\hline ".format(l2_order))
    for i, item in enumerate(err_list):
        if i == 0: 
            print("{} \t\t & {:.6f} &\t\t *  \\\ \hline  \n".format(neuron_nums[i],item) )   
        else: 
            print("{} \t\t &  {:.3e} &  \t\t {:.2f} \\\ \hline  \n".format(neuron_nums[i],item,np.log(err_list[i-1]/err_list[i])/np.log(2) ) )

def show_convergence_order_latex_2(err_l2_list,neuron_num_list ,k=1,d=1): 

#     err_list2 = [err_h10[i] for i in neuron_nums ] 
    # f_write.write('M:{}, relu {} \n'.format(M,k))
    # f_write.write('randomized dictionary size: {}\n'.format(N))
    # f_write.write("neuron num \t\t error \t\t order \t\t h10 error \\ order \n")
    l2_order = -1/2-(2*k + 1)/(2*d)
#     h10_order = -1/2-(2*(k-1) + 1)/(2*d)
#     print("neuron num  & \t $\|u-u_n \|_{L^2}$ & \t order $O(n^{{{}})$ & \t $ | u -u_n |_{H^1}$ & \t order $O(n^{{{}})$ \\\ \hline \hline ".format(l2_order,h10_order))
    print("neuron num  & \t $\\|u-u_n \\|_{{L^2}}$ & \t order $O(n^{{{:.2f}}})$  \\\\ \\hline \\hline ".format(l2_order))
    for i, item in enumerate(err_l2_list):
        if i == 0: 
            # print(neuron_nums[i], end = "\t\t")
            # print(item, end = "\t\t")

            # print("*")
            print("{} \t\t & {:.6f} &\t\t *  \\\ \hline  \n".format(neuron_num_list[i],item) )   
            # f_write.write("{} \t\t {} \t\t * \t\t {} \t\t * \n".format(neuron_nums[i],item, err_list2[i] ))
        else: 
            # print(neuron_nums[i], end = "\t\t")
            # print(item, end = "\t\t") 
            # print(np.log(err_list[i-1]/err_list[i])/np.log(2))
            print("{} \t\t &  {:.3e} &  \t\t {:.2f} \\\ \hline  \n".format(neuron_num_list[i],item,np.log(err_l2_list[i-1]/err_l2_list[i])/np.log(2) ) )
            # f_write.write("{} \t\t {} \t\t {} \t\t {} \t\t {} \n".format(neuron_nums[i],item,np.log(err_list[i-1]/err_list[i])/np.log(2),err_list2[i] , np.log(err_list2[i-1]/err_list2[i])/np.log(2) ))
    # f_write.write("\n")
    # f_write.close()


## OGA 

In [5]:

def target(x): ## Gaussian function in dimension 5   
    d = 5  
    cn =   7.03/d 
    return  torch.exp(-torch.sum( cn**2 * (x/2)**2,dim = 1, keepdim = True)) 

dim = 5 
function_name = "5DGaussian"
filename_write = "data/5D-OGA-{}-order.txt".format(function_name)
M = int(1e6) 
f_write = open(filename_write, "a")
f_write.write("Integration points: Quasi Monte Carlo:  {}\n".format(M))
f_write.close() 
save = False 
write2file = False 
memory = 2**27
for relu_k in [4]: 
    s = 1 
    for N0 in [2**10]: 

        N = s*N0 
        exponent = 9  
        num_epochs=  2**exponent 
        my_model = None 

        err, my_model = OGAL2FittingReLU4Dplus_QMC(my_model,target, \
                    s,N0,num_epochs, M, k = relu_k, linear_solver = "direct", memory=memory)

        if save: 
            folder = 'data/'
            filename = folder + function_name + "_err_randDict_relu_{}_size_{}_num_neurons_{}.pt".format(relu_k,s * N0,num_epochs)
            torch.save(err,filename)
            filename = folder + function_name +  "_model_randDict_relu_{}_size_{}_num_neurons_{}.pt".format(relu_k,s * N0,num_epochs)
            torch.save(my_model.state_dict(),filename) 

        show_convergence_order(err,exponent,N,filename_write,write2file = write2file)
        show_convergence_order_latex(err,exponent,k=relu_k,d=dim)



generate sob sequence: 0.9516849517822266
using linear solver:  direct
epoch:  1	argmax batch num,  8
argmax time taken,  0.6511011123657227
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0006077289581298828
solving Ax = b time taken:  0.126420259475708
		 time taken minimize linear layer:  0.12709569931030273
cuda:0 cuda:0
current error:  tensor(0.7369)
epoch:  2	argmax batch num,  8
argmax time taken,  0.6041193008422852
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0006017684936523438
solving Ax = b time taken:  0.00962686538696289
		 time taken minimize linear layer:  0.01028585433959961
cuda:0 cuda:0
current error:  tensor(0.7310)
epoch:  3	argmax batch num,  8
argmax time taken,  0.6200032234191895
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005006790161132812
solving Ax = b time taken:  0.010254859924316406
		 time taken minimize linear layer:  0.01081228256225586
cuda:0 cuda:0
current error:  tensor(0.7272)
epoch:  4	argma

argmax time taken,  0.6357178688049316
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005090236663818359
solving Ax = b time taken:  0.02881646156311035
		 time taken minimize linear layer:  0.0293881893157959
cuda:0 cuda:0
current error:  tensor(0.6524)
epoch:  29	argmax batch num,  8
argmax time taken,  0.6421871185302734
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005729198455810547
solving Ax = b time taken:  0.02995443344116211
		 time taken minimize linear layer:  0.03059697151184082
cuda:0 cuda:0
current error:  tensor(0.6503)
epoch:  30	argmax batch num,  8
argmax time taken,  0.6302103996276855
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005645751953125
solving Ax = b time taken:  0.030555248260498047
		 time taken minimize linear layer:  0.031181812286376953
cuda:0 cuda:0
current error:  tensor(0.6486)
epoch:  31	argmax batch num,  8
argmax time taken,  0.6250827312469482
num batch:  1
jac:  cuda:0
assembling the mat

argmax time taken,  0.6466050148010254
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005102157592773438
solving Ax = b time taken:  0.06001687049865723
		 time taken minimize linear layer:  0.0605928897857666
cuda:0 cuda:0
current error:  tensor(0.5786)
epoch:  56	argmax batch num,  8
argmax time taken,  0.6471493244171143
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.000530242919921875
solving Ax = b time taken:  0.059766530990600586
		 time taken minimize linear layer:  0.06036543846130371
cuda:0 cuda:0
current error:  tensor(0.5638)
epoch:  57	argmax batch num,  8
argmax time taken,  0.6473581790924072
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005974769592285156
solving Ax = b time taken:  0.06096792221069336
		 time taken minimize linear layer:  0.06163167953491211
cuda:0 cuda:0
current error:  tensor(0.5574)
epoch:  58	argmax batch num,  8
argmax time taken,  0.648557186126709
num batch:  1
jac:  cuda:0
assembling the mat

argmax time taken,  0.6706295013427734
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005946159362792969
solving Ax = b time taken:  0.14298558235168457
		 time taken minimize linear layer:  0.14365124702453613
cuda:0 cuda:0
current error:  tensor(0.4751)
epoch:  83	argmax batch num,  8
argmax time taken,  0.678229570388794
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005571842193603516
solving Ax = b time taken:  0.14380764961242676
		 time taken minimize linear layer:  0.14443612098693848
cuda:0 cuda:0
current error:  tensor(0.4633)
epoch:  84	argmax batch num,  8
argmax time taken,  0.6721172332763672
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0006189346313476562
solving Ax = b time taken:  0.14081311225891113
		 time taken minimize linear layer:  0.14150285720825195
cuda:0 cuda:0
current error:  tensor(0.4616)
epoch:  85	argmax batch num,  8
argmax time taken,  0.6592793464660645
num batch:  1
jac:  cuda:0
assembling the ma

argmax time taken,  0.6767876148223877
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005860328674316406
solving Ax = b time taken:  0.16027283668518066
		 time taken minimize linear layer:  0.16092896461486816
cuda:0 cuda:0
current error:  tensor(0.3766)
epoch:  110	argmax batch num,  8
argmax time taken,  0.6774921417236328
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005974769592285156
solving Ax = b time taken:  0.16045546531677246
		 time taken minimize linear layer:  0.16112279891967773
cuda:0 cuda:0
current error:  tensor(0.3748)
epoch:  111	argmax batch num,  8
argmax time taken,  0.668813943862915
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005993843078613281
solving Ax = b time taken:  0.1611652374267578
		 time taken minimize linear layer:  0.1618340015411377
cuda:0 cuda:0
current error:  tensor(0.3709)
epoch:  112	argmax batch num,  8
argmax time taken,  0.6728076934814453
num batch:  1
jac:  cuda:0
assembling the m

...

argmax time taken,  0.819303035736084
num batch:  3
jac:  cuda:0
assembling the matrix time taken:  0.0017375946044921875
solving Ax = b time taken:  0.9966220855712891
		 time taken minimize linear layer:  0.9985008239746094
cuda:0 cuda:0
current error:  tensor(0.0875)
epoch:  380	argmax batch num,  8
argmax time taken,  0.8193328380584717
num batch:  3
jac:  cuda:0
assembling the matrix time taken:  0.0016129016876220703
solving Ax = b time taken:  0.9975295066833496
		 time taken minimize linear layer:  0.9995121955871582
cuda:0 cuda:0
current error:  tensor(0.0872)
epoch:  381	argmax batch num,  8
argmax time taken,  0.8200597763061523
num batch:  3
jac:  cuda:0
assembling the matrix time taken:  0.0016281604766845703
solving Ax = b time taken:  0.9979417324066162
		 time taken minimize linear layer:  0.9996953010559082
cuda:0 cuda:0
current error:  tensor(0.0867)
epoch:  382	argmax batch num,  8
argmax time taken,  0.820420503616333
num batch:  3
jac:  cuda:0
assembling the matrix

argmax time taken,  0.8353872299194336
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.0016129016876220703
solving Ax = b time taken:  1.2910773754119873
		 time taken minimize linear layer:  1.292849063873291
cuda:0 cuda:0
current error:  tensor(0.0759)
epoch:  407	argmax batch num,  8
argmax time taken,  0.8358542919158936
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.0016665458679199219
solving Ax = b time taken:  1.2913587093353271
		 time taken minimize linear layer:  1.2931499481201172
cuda:0 cuda:0
current error:  tensor(0.0757)
epoch:  408	argmax batch num,  8
argmax time taken,  0.8367161750793457
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.00179290771484375
solving Ax = b time taken:  1.2906467914581299
		 time taken minimize linear layer:  1.2925631999969482
cuda:0 cuda:0
current error:  tensor(0.0754)
epoch:  409	argmax batch num,  8
argmax time taken,  0.8358378410339355
num batch:  4
jac:  cuda:0
assembling the matrix 

argmax time taken,  0.8481087684631348
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.001676797866821289
solving Ax = b time taken:  1.3066036701202393
		 time taken minimize linear layer:  1.3084053993225098
cuda:0 cuda:0
current error:  tensor(0.0688)
epoch:  434	argmax batch num,  8
argmax time taken,  0.8493239879608154
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.0017037391662597656
solving Ax = b time taken:  1.3073928356170654
		 time taken minimize linear layer:  1.3092222213745117
cuda:0 cuda:0
current error:  tensor(0.0685)
epoch:  435	argmax batch num,  8
argmax time taken,  0.855323076248169
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.0017592906951904297
solving Ax = b time taken:  1.3079514503479004
		 time taken minimize linear layer:  1.3098328113555908
cuda:0 cuda:0
current error:  tensor(0.0683)
epoch:  436	argmax batch num,  8
argmax time taken,  0.849928617477417
num batch:  4
jac:  cuda:0
assembling the matrix 

argmax time taken,  0.8704173564910889
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.0019326210021972656
solving Ax = b time taken:  1.6340155601501465
		 time taken minimize linear layer:  1.6360719203948975
cuda:0 cuda:0
current error:  tensor(0.0623)
epoch:  461	argmax batch num,  8
argmax time taken,  0.8713405132293701
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.0017304420471191406
solving Ax = b time taken:  1.6349210739135742
		 time taken minimize linear layer:  1.6367709636688232
cuda:0 cuda:0
current error:  tensor(0.0622)
epoch:  462	argmax batch num,  8
argmax time taken,  0.859412670135498
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.0017673969268798828
solving Ax = b time taken:  1.635432243347168
		 time taken minimize linear layer:  1.6373250484466553
cuda:0 cuda:0
current error:  tensor(0.0620)
epoch:  463	argmax batch num,  8
argmax time taken,  0.8601949214935303
num batch:  4
jac:  cuda:0
assembling the matrix

argmax time taken,  0.8777000904083252
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.0017502307891845703
solving Ax = b time taken:  1.6512022018432617
		 time taken minimize linear layer:  1.6530795097351074
cuda:0 cuda:0
current error:  tensor(0.0581)
epoch:  488	argmax batch num,  8
argmax time taken,  0.8782916069030762
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.0018219947814941406
solving Ax = b time taken:  1.6502234935760498
		 time taken minimize linear layer:  1.6521790027618408
cuda:0 cuda:0
current error:  tensor(0.0580)
epoch:  489	argmax batch num,  8
argmax time taken,  0.8725118637084961
num batch:  4
jac:  cuda:0
assembling the matrix time taken:  0.0018537044525146484
solving Ax = b time taken:  1.6522858142852783
		 time taken minimize linear layer:  1.6542627811431885
cuda:0 cuda:0
current error:  tensor(0.0579)
epoch:  490	argmax batch num,  8
argmax time taken,  0.8788671493530273
num batch:  4
jac:  cuda:0
assembling the matr

In [33]:
def compute_l2_error(u_exact,my_model,M,batch_size_2,weights,integration_points): 
    err = 0 
    if my_model == None: 
        for jj in range(0,M,batch_size_2): 
            end_index = jj + batch_size_2 
            func_values = u_exact(integration_points[jj:end_index,:])
            err += torch.sum(func_values**2 * weights[jj:end_index,:]) # **0.5
    else: 
        for jj in range(0,M,batch_size_2): 
            end_index = jj + batch_size_2 
            func_values = u_exact(integration_points[jj:end_index,:]) - my_model(integration_points[jj:end_index,:]).detach()
            err += torch.sum(func_values**2 * weights[jj:end_index,:])# **0.5	
    return err**0.5  

def target(x): ## Gaussian function in dimension 5   
    d = 5  
    cn =   7.03/d 
    return  torch.exp(-torch.sum( cn**2 * (x/2)**2,dim = 1, keepdim = True)) 

relu_k = 4   
M = int(1e6)
integration_weights, integration_points = MonteCarlo_Sobol_dDim_weights_points(M ,d = 5,bl = -1,ur = 1)
num_trials = 5 
neuron_num_list = [16,32,64,128,512,1024,2048]
err_trials = torch.zeros(len(neuron_num_list), num_trials)
for i, neuron_num in enumerate(neuron_num_list): 
    print("======> ",neuron_num)
    for j in range(num_trials): 
        my_model = model(5, neuron_num, 1, k = relu_k).to(device) 
        my_model = initialize_model_random(my_model.cpu(),dim = 5)
        my_model = remove_redundant_neuron(my_model.cpu(), dims = 5, choice = 2).to(device)
        sol = minimize_linear_layer_explicit_assemble(my_model,target,integration_weights, integration_points,solver="direct",memory=2**27)
        my_model.fc2.weight.data[0,:] = sol[:]

        memory = 2**27 
        total_size = neuron_num * M # memory, number of floating numbers 
        num_batch = total_size//memory + 1 # divide according to memory
        batch_size = M//num_batch
    #     with torch.no_grad(): 
        err_l2 =compute_l2_error(target,my_model,M,batch_size,integration_weights,integration_points)
    #         func_diff_sqrd = (my_model(integration_points) - target(integration_points))**2 
    #     err_l2 = (integration_weights.t() @ func_diff_sqrd )**0.5 
        err_trials[i,j] = err_l2
        print("l2 err: ",err_l2)


======>  16
Number of neurons removed:  0
Number of neurons left:  16
after replenishing neurons:  16
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0006635189056396484
solving Ax = b time taken:  0.02217721939086914
l2 err:  tensor(0.7259, device='cuda:0')
Number of neurons removed:  0
Number of neurons left:  16
after replenishing neurons:  16
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005035400390625
solving Ax = b time taken:  0.022368192672729492
l2 err:  tensor(0.7136, device='cuda:0')
Number of neurons removed:  0
Number of neurons left:  16
after replenishing neurons:  16
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0005764961242675781
solving Ax = b time taken:  0.02245640754699707
l2 err:  tensor(0.7176, device='cuda:0')
Number of neurons removed:  1
Number of neurons left:  15
after replenishing neurons:  16
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.00048732757568359375
solving Ax = b time taken:

l2 err:  tensor(0.0220, device='cuda:0')
Number of neurons removed:  9
Number of neurons left:  2039
after replenishing neurons:  2048
num batch:  16
jac:  cuda:0
assembling the matrix time taken:  0.014692544937133789
solving Ax = b time taken:  23.255043029785156
l2 err:  tensor(0.0215, device='cuda:0')
Number of neurons removed:  22
Number of neurons left:  2026
after replenishing neurons:  2048
num batch:  16
jac:  cuda:0
assembling the matrix time taken:  0.01111459732055664
solving Ax = b time taken:  23.259803533554077
l2 err:  tensor(0.0223, device='cuda:0')
Number of neurons removed:  8
Number of neurons left:  2040
after replenishing neurons:  2048
num batch:  16
jac:  cuda:0
assembling the matrix time taken:  0.01823115348815918
solving Ax = b time taken:  23.262007474899292
l2 err:  tensor(0.0224, device='cuda:0')
Number of neurons removed:  17
Number of neurons left:  2031
after replenishing neurons:  2048
num batch:  16
jac:  cuda:0
assembling the matrix time taken:  0.01

In [36]:
err_mean = err_trials.mean(dim = 1)
print(err_mean)
err_l2_list = [err.data for err in err_mean]
print(err_l2_list)
show_convergence_order_latex_2(err_l2_list,neuron_num_list ,k=relu_k,d=5)



tensor([0.7210, 0.6997, 0.6636, 0.5684, 0.1675, 0.0492, 0.0222])
[tensor(0.7210), tensor(0.6997), tensor(0.6636), tensor(0.5684), tensor(0.1675), tensor(0.0492), tensor(0.0222)]
neuron num  & 	 $\|u-u_n \|_{L^2}$ & 	 order $O(n^{-1.40})$  \\ \hline \hline 
16 		 & 0.721009 &		 *  \\ \hline  

32 		 &  6.997e-01 &  		 0.04 \\ \hline  

64 		 &  6.636e-01 &  		 0.08 \\ \hline  

128 		 &  5.684e-01 &  		 0.22 \\ \hline  

512 		 &  1.675e-01 &  		 1.76 \\ \hline  

1024 		 &  4.918e-02 &  		 1.77 \\ \hline  

2048 		 &  2.215e-02 &  		 1.15 \\ \hline  



neuron num  & 	 $\|u-u_n \|_{L^2}$ & 	 order $O(n^{-1.00})$  \\ \hline \hline 
16 		 & 0.644036 &		 *  \\ \hline  

32 		 &  5.443e-01 &  		 0.24 \\ \hline  

64 		 &  3.879e-01 &  		 0.49 \\ \hline  

128 		 &  2.767e-01 &  		 0.49 \\ \hline  

512 		 &  9.455e-02 &  		 1.55 \\ \hline  

1024 		 &  6.139e-02 &  		 0.62 \\ \hline  

2048 		 &  3.778e-02 &  		 0.70 \\ \hline  



In [15]:
def compute_l2_error(u_exact,my_model,M,batch_size_2,weights,integration_points): 
    err = 0 
    if my_model == None: 
        for jj in range(0,M,batch_size_2): 
            end_index = jj + batch_size_2 
            func_values = u_exact(integration_points[jj:end_index,:])
            err += torch.sum(func_values**2 * weights[jj:end_index,:]) # **0.5
    else: 
        for jj in range(0,M,batch_size_2): 
            end_index = jj + batch_size_2 
            func_values = u_exact(integration_points[jj:end_index,:]) - my_model(integration_points[jj:end_index,:]).detach()
            err += torch.sum(func_values**2 * weights[jj:end_index,:])# **0.5	
    return err**0.5  

def target(x): ## Gaussian function in dimension 5   

#     return torch.prod(torch.sin(pi/4 * x),dim = 1, keepdim = True )
    return torch.cos(pi/4 * torch.sum(x,dim = 1, keepdim = True))

relu_k = 2   
M = int(1e6)
integration_weights, integration_points = MonteCarlo_Sobol_dDim_weights_points(M ,d = 5,bl = -1,ur = 1)
for neuron_num in [16,32,64,128,512,1024,2048]: 
    
    my_model = model(5, neuron_num, 1, k = relu_k).to(device) 
    my_model = initialize_model_random(my_model.cpu(),dim = 5)
    my_model = remove_redundant_neuron(my_model.cpu(), dims = 5, choice = 2).to(device)
    sol = minimize_linear_layer_explicit_assemble(my_model,target,integration_weights, integration_points,solver="direct",memory=2**27)
    my_model.fc2.weight.data[0,:] = sol[:]

    memory = 2**28 
    total_size = neuron_num * M # memory, number of floating numbers 
    num_batch = total_size//memory + 1 # divide according to memory
    batch_size = M//num_batch
#     with torch.no_grad(): 
    err_l2 =compute_l2_error(target,my_model,M,batch_size,integration_weights,integration_points)
#         func_diff_sqrd = (my_model(integration_points) - target(integration_points))**2 
#     err_l2 = (integration_weights.t() @ func_diff_sqrd )**0.5 
    print("l2 err: ",err_l2)
    

Number of neurons removed:  1
Number of neurons left:  99
num batch:  1
jac:  cuda:0
assembling the matrix time taken:  0.0006206035614013672
solving Ax = b time taken:  0.189192533493042
l2 err:  tensor(2.6677, device='cuda:0')
Number of neurons removed:  0
Number of neurons left:  200
num batch:  2
jac:  cuda:0
assembling the matrix time taken:  0.0009438991546630859
solving Ax = b time taken:  0.48268651962280273
l2 err:  tensor(1.6896, device='cuda:0')
Number of neurons removed:  1
Number of neurons left:  399
num batch:  3
jac:  cuda:0
assembling the matrix time taken:  0.001535177230834961
solving Ax = b time taken:  1.279845952987671
l2 err:  tensor(0.8292, device='cuda:0')
Number of neurons removed:  5
Number of neurons left:  795
num batch:  6
jac:  cuda:0
assembling the matrix time taken:  0.0026030540466308594
solving Ax = b time taken:  4.014509916305542
l2 err:  tensor(0.2249, device='cuda:0')
Number of neurons removed:  12
Number of neurons left:  1588
num batch:  12
jac:

In [4]:
def compute_l2_error(u_exact,my_model,M,batch_size_2,weights,integration_points): 
    err = 0 
    if my_model == None: 
        for jj in range(0,M,batch_size_2): 
            end_index = jj + batch_size_2 
            func_values = u_exact(integration_points[jj:end_index,:])
            err += torch.sum(func_values**2 * weights[jj:end_index,:]) # **0.5
    else: 
        for jj in range(0,M,batch_size_2): 
            end_index = jj + batch_size_2 
            func_values = u_exact(integration_points[jj:end_index,:]) - my_model(integration_points[jj:end_index,:]).detach()
            err += torch.sum(func_values**2 * weights[jj:end_index,:])# **0.5	
    return err**0.5  


def target(x): ## Gaussian function in dimension 5   
    return torch.sum((x)**3,dim = 1, keepdim = True) 

relu_k = 2   
M = int(1e6)
integration_weights, integration_points = MonteCarlo_Sobol_dDim_weights_points(M ,d = 5,bl = -1,ur = 1)
for neuron_num in [200,400,800,1600]: 
    
    my_model = model(5, neuron_num, 1, k = relu_k).to(device) 
    my_model = initialize_model_random(my_model.cpu(),dim = 5)
    my_model = remove_redundant_neuron(my_model.cpu(), dims = 5, choice = 2).to(device)
    sol = minimize_linear_layer_explicit_assemble(my_model,target,integration_weights, integration_points,solver="direct",memory=2**27)
    my_model.fc2.weight.data[0,:] = sol[:]

    memory = 2**27
    total_size = neuron_num * M # memory, number of floating numbers 
    num_batch = total_size//memory + 1 # divide according to memory
    batch_size = M//num_batch
#     with torch.no_grad(): 
    err_l2 =compute_l2_error(target,my_model,M,batch_size,integration_weights,integration_points)
#         func_diff_sqrd = (my_model(integration_points) - target(integration_points))**2 
#     err_l2 = (integration_weights.t() @ func_diff_sqrd )**0.5 
    print("l2 err: ",err_l2)
    

Number of neurons removed:  0
Number of neurons left:  200
num batch:  2
jac:  cuda:0
assembling the matrix time taken:  0.0008752346038818359
solving Ax = b time taken:  0.4037768840789795
l2 err:  tensor(2.8039, device='cuda:0')
Number of neurons removed:  0
Number of neurons left:  400
num batch:  3
jac:  cuda:0
assembling the matrix time taken:  0.0013287067413330078
solving Ax = b time taken:  1.1068832874298096
l2 err:  tensor(1.6944, device='cuda:0')
Number of neurons removed:  3
Number of neurons left:  797
num batch:  6
jac:  cuda:0
assembling the matrix time taken:  0.00267791748046875
solving Ax = b time taken:  3.6753299236297607
l2 err:  tensor(0.9390, device='cuda:0')
Number of neurons removed:  8
Number of neurons left:  1592
num batch:  12
jac:  cuda:0
assembling the matrix time taken:  0.006819963455200195
solving Ax = b time taken:  14.120642185211182
l2 err:  tensor(0.5638, device='cuda:0')


In [33]:
## test accuracy:  
M2 = 10000
integration_weights, integration_points = MonteCarlo_Sobol_dDim_weights_points(M2 ,d = 5,bl = -1,ur =1)

# print(my_model(integration_points))
# print(target(integration_points))
with torch.no_grad():
#     func_diff_sqrd = (target(integration_points) - my_model(integration_points))**2
#     test_err = (integration_weights.t() @ func_diff_sqrd)**0.5 
    func_diff_abs = torch.abs(target(integration_points) - my_model(integration_points))
    test_err = torch.max(func_diff_abs)
    print(torch.max(target(integration_points)))
    
print(test_err)

tensor(25.9815, device='cuda:0')
tensor(0.8589, device='cuda:0')


In [10]:
my_model.fc2.weight.data

tensor([[-0.0216,  0.0052, -0.0138,  ...,  0.0108,  0.0613, -0.0700]],
       device='cuda:0')